# Forecasting hourly OTTO cart and order volume with a Transformer

I built this experiment to test whether a small Transformer can learn short-term demand patterns from the OTTO session logs. I first collapse the raw click, cart and order events into an hourly time series, then use the previous seven days to forecast the next 24 hours of cart and order activity.

The Transformer is intentionally small because the aggregated series is short. I compare it with persistence, daily and weekly seasonal baselines, Ridge regression and Extra Trees so the deep-learning result has some context instead of being evaluated on its own.

## Setup

I use a GPU for the Transformer training. The data aggregation itself is CPU-bound.

In [ ]:
!pip -q install --upgrade kagglehub pandas scikit-learn matplotlib joblib

In [ ]:
import copy
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

print("Python ready")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Working folders

I keep the processed hourly series and experiment outputs in Drive so I do not have to rebuild everything after a Colab runtime resets.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/OTTO_Dissertation")
RAW_DRIVE_DIR = DRIVE_ROOT / "data" / "raw"
PROCESSED_DRIVE_DIR = DRIVE_ROOT / "data" / "processed"
ARTIFACT_DRIVE_DIR = DRIVE_ROOT / "artifacts" / "transformer_experiment"

for folder in [
    RAW_DRIVE_DIR,
    PROCESSED_DRIVE_DIR,
    ARTIFACT_DRIVE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path("/content/otto_transformer_experiment")
LOCAL_RAW_DIR = LOCAL_ROOT / "data" / "raw"
LOCAL_PROCESSED_DIR = LOCAL_ROOT / "data" / "processed"

for folder in [LOCAL_RAW_DIR, LOCAL_PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Artifact folder:", ARTIFACT_DRIVE_DIR)

## Experiment settings

The main forecasting setup uses one week (168 hours) of history to predict the next day (24 hours). The final four days are held out for testing, with the four days before that used for validation.

In [ ]:
SEED = 42

LOOKBACK = 168
HORIZON = 24
VALIDATION_STEPS = 96
TEST_STEPS = 96

BATCH_SIZE = 32
EPOCHS = 120
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
PATIENCE = 15
GRADIENT_CLIP = 1.0

D_MODEL = 32
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.15

RIDGE_ALPHAS = (0.1, 1.0, 10.0, 100.0)
EXTRA_TREES_ESTIMATORS = 500
EXTRA_TREES_MIN_SAMPLES_LEAF = 3
EXTRA_TREES_MAX_FEATURES = 0.7

EVENT_TYPES = ("clicks", "carts", "orders")
HISTORY_FEATURE_COLUMNS = ("log_clicks", "log_carts", "log_orders")
TEMPORAL_FEATURE_COLUMNS = (
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_weekend",
)
FEATURE_COLUMNS = HISTORY_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS
TARGET_COLUMNS = ("carts", "orders")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Features:", FEATURE_COLUMNS)
print("Targets:", TARGET_COLUMNS)

## Load the OTTO data

If I have already built `otto_hourly.csv`, I reuse it. Otherwise I download the OTTO training JSONL from Kaggle and aggregate it once.

In [ ]:
PROCESSED_DRIVE_PATH = PROCESSED_DRIVE_DIR / "otto_hourly.csv"
PROCESSED_LOCAL_PATH = LOCAL_PROCESSED_DIR / "otto_hourly.csv"
RAW_DRIVE_PATH = RAW_DRIVE_DIR / "otto-recsys-train.jsonl"
RAW_LOCAL_PATH = LOCAL_RAW_DIR / "otto-recsys-train.jsonl"

print("Processed file already in Drive:", PROCESSED_DRIVE_PATH.exists())
print("Raw file already in Drive:", RAW_DRIVE_PATH.exists())

In [ ]:
# Reuse the processed series when it is already available.

if PROCESSED_DRIVE_PATH.exists():
    print("Processed CSV already exists. Kaggle download is not needed.")
else:
    if RAW_DRIVE_PATH.exists():
        print("Using raw JSONL already stored in Drive.")
        shutil.copy2(RAW_DRIVE_PATH, RAW_LOCAL_PATH)
    else:
        import kagglehub

        try:
            downloaded = kagglehub.dataset_download(
                "otto/recsys-dataset",
                path="otto-recsys-train.jsonl",
                output_dir=str(LOCAL_RAW_DIR),
            )
        except Exception as exc:
            print("Kaggle authentication is required.")
            print("Original error:", exc)
            kagglehub.login()
            downloaded = kagglehub.dataset_download(
                "otto/recsys-dataset",
                path="otto-recsys-train.jsonl",
                output_dir=str(LOCAL_RAW_DIR),
                force_download=True,
            )

        downloaded_path = Path(downloaded)

        candidates = []
        if downloaded_path.is_file():
            candidates.append(downloaded_path)
        if downloaded_path.is_dir():
            candidates.extend(downloaded_path.rglob("otto-recsys-train.jsonl"))
        candidates.extend(LOCAL_RAW_DIR.rglob("otto-recsys-train.jsonl"))

        candidates = list(dict.fromkeys(candidates))
        if not candidates:
            raise FileNotFoundError(
                "Kaggle download completed, but otto-recsys-train.jsonl was not found."
            )

        source = candidates[0]
        if source.resolve() != RAW_LOCAL_PATH.resolve():
            shutil.copy2(source, RAW_LOCAL_PATH)

    print("Raw dataset ready:", RAW_LOCAL_PATH)
    print("Size (GB):", round(RAW_LOCAL_PATH.stat().st_size / 1_000_000_000, 2))

## Build the hourly series

The raw file is too large to load into memory comfortably, so I read it session by session and count clicks, carts and orders in one-hour UTC buckets. Any missing hour is inserted with zero counts.

In [ ]:
def parse_fixed_frequency(frequency: str) -> pd.Timedelta:
    try:
        offset = pd.tseries.frequencies.to_offset(frequency)
        delta = pd.Timedelta(offset.nanos, unit="ns")
    except (TypeError, ValueError) as exc:
        raise ValueError(
            "Frequency must be a fixed duration such as '1h' or '30min'"
        ) from exc
    if delta <= pd.Timedelta(0):
        raise ValueError("Frequency must be positive")
    return delta


def validate_hourly_frame(frame: pd.DataFrame, frequency: str = "1h") -> None:
    required = {"timestamp", *EVENT_TYPES}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError("The processed dataset is empty")

    timestamps = pd.to_datetime(frame["timestamp"], utc=True, errors="raise")
    if timestamps.duplicated().any():
        raise ValueError("Duplicate timestamps were found")
    if not timestamps.is_monotonic_increasing:
        raise ValueError("Timestamps must be in chronological order")
    if len(timestamps) > 1:
        expected = parse_fixed_frequency(frequency)
        if not timestamps.diff().dropna().eq(expected).all():
            raise ValueError(
                f"Timestamps must be consecutive at frequency {frequency}"
            )

    event_values = frame.loc[:, EVENT_TYPES].to_numpy(dtype=np.float64)
    if not np.isfinite(event_values).all():
        raise ValueError("Event columns contain non-finite values")
    if (event_values < 0).any():
        raise ValueError("Event counts cannot be negative")


def aggregate_jsonl(
    input_path: str | Path,
    frequency: str = "1h",
    progress_every: int = 1_000_000,
) -> pd.DataFrame:
    source = Path(input_path)
    if not source.exists():
        raise FileNotFoundError(f"Dataset not found: {source}")

    interval = parse_fixed_frequency(frequency)
    interval_ms = int(interval.total_seconds() * 1000)

    counts: dict[int, dict[str, int]] = defaultdict(
        lambda: {event_type: 0 for event_type in EVENT_TYPES}
    )

    started = time.perf_counter()

    with source.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            try:
                session = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on line {line_number}") from exc

            for event in session.get("events", []):
                event_type = event.get("type")
                timestamp = event.get("ts")

                if event_type not in EVENT_TYPES or timestamp is None:
                    continue

                bucket = int(timestamp) // interval_ms * interval_ms
                counts[bucket][event_type] += 1

            if progress_every and line_number % progress_every == 0:
                elapsed = time.perf_counter() - started
                print(
                    f"Processed {line_number:,} sessions "
                    f"in {elapsed / 60:.1f} minutes"
                )

    if not counts:
        raise ValueError("No supported OTTO events were found")

    rows = [
        {"timestamp": timestamp, **event_counts}
        for timestamp, event_counts in counts.items()
    ]

    frame = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], unit="ms", utc=True)

    full_range = pd.date_range(
        start=frame["timestamp"].min(),
        end=frame["timestamp"].max(),
        freq=frequency,
        tz="UTC",
    )

    frame = (
        frame.set_index("timestamp")
        .reindex(full_range, fill_value=0)
        .rename_axis("timestamp")
        .reset_index()
    )

    validate_hourly_frame(frame, frequency)
    return frame

In [ ]:
if PROCESSED_DRIVE_PATH.exists():
    shutil.copy2(PROCESSED_DRIVE_PATH, PROCESSED_LOCAL_PATH)
    hourly = pd.read_csv(PROCESSED_LOCAL_PATH)
    hourly["timestamp"] = pd.to_datetime(hourly["timestamp"], utc=True)
    validate_hourly_frame(hourly)
    print("Loaded processed hourly dataset from Drive.")
else:
    if not RAW_LOCAL_PATH.exists():
        raise FileNotFoundError(
            "Raw dataset is not available. Run the Kaggle download cell first."
        )

    hourly = aggregate_jsonl(RAW_LOCAL_PATH, frequency="1h")
    hourly.to_csv(PROCESSED_LOCAL_PATH, index=False)
    shutil.copy2(PROCESSED_LOCAL_PATH, PROCESSED_DRIVE_PATH)
    print("Hourly aggregation completed and saved to Drive.")

print("Shape:", hourly.shape)
display(hourly.head())
display(hourly.tail())

## Quick data check

Before modeling, I check the time range, event totals, missing hours and a SHA-256 fingerprint of the processed CSV. The hash makes it easier to verify that later results came from the same data file.

In [ ]:
def sha256_file(path: str | Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def audit_frame(
    frame: pd.DataFrame,
    source_path: str | Path | None = None,
) -> dict[str, Any]:
    validate_hourly_frame(frame)
    timestamps = pd.to_datetime(frame["timestamp"], utc=True)
    event_values = frame.loc[:, EVENT_TYPES]

    audit: dict[str, Any] = {
        "rows": int(len(frame)),
        "start_timestamp": timestamps.iloc[0].isoformat(),
        "end_timestamp": timestamps.iloc[-1].isoformat(),
        "duration_hours": int(len(frame)),
        "frequency": "1h",
        "duplicate_timestamps": int(timestamps.duplicated().sum()),
        "zero_activity_hours": int((event_values.sum(axis=1) == 0).sum()),
        "event_totals": {
            column: int(event_values[column].sum())
            for column in EVENT_TYPES
        },
        "event_means": {
            column: float(event_values[column].mean())
            for column in EVENT_TYPES
        },
        "event_maxima": {
            column: int(event_values[column].max())
            for column in EVENT_TYPES
        },
    }

    if source_path is not None and Path(source_path).exists():
        audit["source_path"] = str(source_path)
        audit["sha256"] = sha256_file(source_path)
        audit["size_bytes"] = int(Path(source_path).stat().st_size)

    return audit


audit = audit_frame(hourly, PROCESSED_LOCAL_PATH)

print(json.dumps(audit, indent=2))
display(hourly[list(EVENT_TYPES)].describe())

(ARTIFACT_DRIVE_DIR / "data_audit.json").write_text(
    json.dumps(audit, indent=2),
    encoding="utf-8",
)

In [ ]:
figure = plt.figure(figsize=(14, 8))

for index, column in enumerate(EVENT_TYPES, start=1):
    axis = figure.add_subplot(3, 1, index)
    axis.plot(hourly["timestamp"], hourly[column])
    axis.set_ylabel(column.title())
    axis.grid(alpha=0.25)

figure.suptitle("OTTO hourly event volumes")
figure.tight_layout()
figure.savefig(ARTIFACT_DRIVE_DIR / "hourly_event_series.png", dpi=180)
plt.show()

## Seasonal pattern check

Before choosing the 24-hour and 168-hour seasonal baselines, I check whether those patterns are
actually visible in the aggregated series. The plots below are descriptive only; they are not
used to tune the final holdout.


In [ ]:
eda = hourly.copy()
eda["timestamp"] = pd.to_datetime(eda["timestamp"], utc=True)
eda["hour"] = eda["timestamp"].dt.hour
eda["day_name"] = eda["timestamp"].dt.day_name()

hour_profile = (
    eda.groupby("hour")[["carts", "orders"]]
    .mean()
)

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]
day_profile = (
    eda.groupby("day_name")[["carts", "orders"]]
    .mean()
    .reindex(day_order)
)

figure = plt.figure(figsize=(10, 5))
axis = figure.add_subplot(111)
axis.plot(hour_profile.index, hour_profile["carts"], marker="o", label="Carts")
axis.plot(hour_profile.index, hour_profile["orders"], marker="o", label="Orders")
axis.set_xlabel("Hour of day (UTC)")
axis.set_ylabel("Mean events")
axis.set_title("Average activity by hour of day")
axis.legend()
axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

figure = plt.figure(figsize=(10, 5))
axis = figure.add_subplot(111)
x = np.arange(len(day_profile))
width = 0.38
axis.bar(x - width / 2, day_profile["carts"], width=width, label="Carts")
axis.bar(x + width / 2, day_profile["orders"], width=width, label="Orders")
axis.set_xticks(x)
axis.set_xticklabels([day[:3] for day in day_profile.index])
axis.set_ylabel("Mean events")
axis.set_title("Average activity by day of week")
axis.legend()
figure.tight_layout()
plt.show()

lag_check = pd.DataFrame(
    {
        "target": ["carts", "orders"],
        "lag_24_autocorrelation": [
            eda["carts"].autocorr(lag=24),
            eda["orders"].autocorr(lag=24),
        ],
        "lag_168_autocorrelation": [
            eda["carts"].autocorr(lag=168),
            eda["orders"].autocorr(lag=168),
        ],
    }
)

display(lag_check)


## Features and chronological split

The count features are log-transformed because the event volumes are strongly skewed. Hour-of-day and day-of-week are encoded cyclically, and I add a weekend flag. Scalers are fitted only on the training period so information from validation or test hours does not leak backwards.

In [ ]:
def add_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    validate_hourly_frame(frame)

    result = frame.copy()
    result["timestamp"] = pd.to_datetime(result["timestamp"], utc=True)

    hour = result["timestamp"].dt.hour.to_numpy()
    day = result["timestamp"].dt.dayofweek.to_numpy()

    result["log_clicks"] = np.log1p(
        result["clicks"].to_numpy(dtype=np.float64)
    )
    result["log_carts"] = np.log1p(
        result["carts"].to_numpy(dtype=np.float64)
    )
    result["log_orders"] = np.log1p(
        result["orders"].to_numpy(dtype=np.float64)
    )

    result["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    result["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    result["day_sin"] = np.sin(2 * np.pi * day / 7)
    result["day_cos"] = np.cos(2 * np.pi * day / 7)
    result["is_weekend"] = (day >= 5).astype(np.float32)

    return result


@dataclass(frozen=True)
class FoldSpec:
    fold_id: int
    train_end: int
    validation_start: int
    test_start: int
    test_end: int


@dataclass(frozen=True)
class PreparedData:
    features: np.ndarray
    targets: np.ndarray
    raw_targets: np.ndarray
    train_starts: np.ndarray
    validation_starts: np.ndarray
    test_starts: np.ndarray
    feature_scaler: StandardScaler
    target_scaler: StandardScaler
    fold: FoldSpec


def make_final_fold(
    length: int,
    lookback: int,
    horizon: int,
    validation_steps: int,
    test_steps: int,
) -> FoldSpec:
    if validation_steps < horizon or test_steps < horizon:
        raise ValueError(
            "Validation and test periods must each be at least one horizon"
        )

    test_start = length - test_steps
    validation_start = test_start - validation_steps

    if validation_start <= lookback:
        raise ValueError(
            "Not enough observations for the requested lookback and splits"
        )

    return FoldSpec(
        fold_id=0,
        train_end=validation_start,
        validation_start=validation_start,
        test_start=test_start,
        test_end=length,
    )


def target_starts_for_fold(
    length: int,
    lookback: int,
    horizon: int,
    fold: FoldSpec,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    all_starts = np.arange(
        lookback,
        length - horizon + 1,
        dtype=np.int64,
    )

    train = all_starts[
        all_starts + horizon <= fold.train_end
    ]

    validation = all_starts[
        (all_starts >= fold.validation_start)
        & (all_starts + horizon <= fold.test_start)
    ]

    test = all_starts[
        (all_starts >= fold.test_start)
        & (all_starts + horizon <= fold.test_end)
    ]

    if not len(train) or not len(validation) or not len(test):
        raise ValueError("One or more dataset splits are empty")

    return train, validation, test


def prepare_arrays(
    feature_values: np.ndarray,
    raw_target_values: np.ndarray,
    lookback: int,
    horizon: int,
    validation_steps: int,
    test_steps: int,
) -> PreparedData:
    fold = make_final_fold(
        length=len(feature_values),
        lookback=lookback,
        horizon=horizon,
        validation_steps=validation_steps,
        test_steps=test_steps,
    )

    train_starts, validation_starts, test_starts = target_starts_for_fold(
        length=len(feature_values),
        lookback=lookback,
        horizon=horizon,
        fold=fold,
    )

    feature_scaler = StandardScaler().fit(
        feature_values[: fold.train_end]
    )

    log_targets = np.log1p(
        raw_target_values.astype(np.float64)
    )

    target_scaler = StandardScaler().fit(
        log_targets[: fold.train_end]
    )

    features = feature_scaler.transform(feature_values)
    targets = target_scaler.transform(log_targets)

    return PreparedData(
        features=features,
        targets=targets,
        raw_targets=raw_target_values.astype(np.float64),
        train_starts=train_starts,
        validation_starts=validation_starts,
        test_starts=test_starts,
        feature_scaler=feature_scaler,
        target_scaler=target_scaler,
        fold=fold,
    )


featured = add_time_features(hourly)

feature_values = featured.loc[
    :, FEATURE_COLUMNS
].to_numpy(dtype=np.float64)

target_values = featured.loc[
    :, TARGET_COLUMNS
].to_numpy(dtype=np.float64)

prepared = prepare_arrays(
    feature_values=feature_values,
    raw_target_values=target_values,
    lookback=LOOKBACK,
    horizon=HORIZON,
    validation_steps=VALIDATION_STEPS,
    test_steps=TEST_STEPS,
)

print("Fold:", prepared.fold)
print("Training windows:", len(prepared.train_starts))
print("Validation windows:", len(prepared.validation_starts))
print("Test windows:", len(prepared.test_starts))

## Sliding windows

Each sample contains 168 historical hours and a 24-hour target window. The model predicts carts and orders together.

In [ ]:
class ForecastDataset(Dataset):
    def __init__(
        self,
        features: np.ndarray,
        targets: np.ndarray,
        target_starts: np.ndarray,
        lookback: int,
        horizon: int,
    ) -> None:
        self.features = features.astype(np.float32, copy=False)
        self.targets = targets.astype(np.float32, copy=False)
        self.target_starts = target_starts.astype(np.int64, copy=False)
        self.lookback = lookback
        self.horizon = horizon

    def __len__(self) -> int:
        return len(self.target_starts)

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        target_start = int(self.target_starts[index])

        x = self.features[
            target_start - self.lookback : target_start
        ]

        y = self.targets[
            target_start : target_start + self.horizon
        ]

        return torch.from_numpy(x), torch.from_numpy(y)


def build_loaders(
    prepared: PreparedData,
    lookback: int,
    horizon: int,
    batch_size: int,
    seed: int,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    train_dataset = ForecastDataset(
        prepared.features,
        prepared.targets,
        prepared.train_starts,
        lookback,
        horizon,
    )

    validation_dataset = ForecastDataset(
        prepared.features,
        prepared.targets,
        prepared.validation_starts,
        lookback,
        horizon,
    )

    test_dataset = ForecastDataset(
        prepared.features,
        prepared.targets,
        prepared.test_starts,
        lookback,
        horizon,
    )

    loader_args = {
        "batch_size": batch_size,
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }

    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=generator,
        **loader_args,
    )

    validation_loader = DataLoader(
        validation_dataset,
        shuffle=False,
        **loader_args,
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **loader_args,
    )

    return train_loader, validation_loader, test_loader


def inverse_targets(
    values: np.ndarray,
    scaler: StandardScaler,
) -> np.ndarray:
    shape = values.shape

    restored = scaler.inverse_transform(
        values.reshape(-1, shape[-1])
    )

    return np.maximum(
        np.expm1(restored),
        0.0,
    ).reshape(shape)


train_loader, validation_loader, test_loader = build_loaders(
    prepared=prepared,
    lookback=LOOKBACK,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

batch_features, batch_targets = next(iter(train_loader))

print("Input batch shape:", tuple(batch_features.shape))
print("Target batch shape:", tuple(batch_targets.shape))

## Transformer model

The model projects the eight input features into a 32-dimensional representation, adds sinusoidal positional encoding, and passes the sequence through two Transformer encoder layers. I use a learned attention-pooling layer to reduce the 168 encoded time steps to one context vector before producing all 24 forecast steps at once.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(
        self,
        d_model: int,
        max_length: int = 4096,
    ) -> None:
        super().__init__()

        positions = torch.arange(
            max_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        frequencies = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )
            * (-math.log(10000.0) / d_model)
        )

        encoding = torch.zeros(
            max_length,
            d_model,
            dtype=torch.float32,
        )

        encoding[:, 0::2] = torch.sin(
            positions * frequencies
        )

        encoding[:, 1::2] = torch.cos(
            positions * frequencies
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
            persistent=False,
        )

    def forward(
        self,
        values: torch.Tensor,
    ) -> torch.Tensor:
        return values + self.encoding[:, : values.size(1)]


class AttentionPooling(nn.Module):
    def __init__(self, d_model: int) -> None:
        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1, bias=False),
        )

    def forward(
        self,
        values: torch.Tensor,
    ) -> torch.Tensor:
        weights = torch.softmax(
            self.score(values),
            dim=1,
        )

        return torch.sum(
            values * weights,
            dim=1,
        )


class DemandTransformer(nn.Module):
    def __init__(
        self,
        num_features: int,
        target_dim: int,
        horizon: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError(
                "d_model must be divisible by nhead"
            )

        self.horizon = horizon
        self.target_dim = target_dim

        self.input_projection = nn.Linear(
            num_features,
            d_model,
        )

        self.position = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.normalization = nn.LayerNorm(d_model)
        self.pooling = AttentionPooling(d_model)

        self.output = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                dim_feedforward,
                horizon * target_dim,
            ),
        )

    def forward(
        self,
        values: torch.Tensor,
    ) -> torch.Tensor:
        encoded = self.input_projection(values)
        encoded = self.position(encoded)
        encoded = self.encoder(encoded)

        pooled = self.pooling(
            self.normalization(encoded)
        )

        forecast = self.output(pooled)

        return forecast.view(
            values.size(0),
            self.horizon,
            self.target_dim,
        )


def count_parameters(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


model = DemandTransformer(
    num_features=len(FEATURE_COLUMNS),
    target_dim=len(TARGET_COLUMNS),
    horizon=HORIZON,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(DEVICE)

with torch.inference_mode():
    example_output = model(
        batch_features[:2].to(DEVICE)
    )

print(model)
print("Trainable parameters:", count_parameters(model))
print("Example output shape:", tuple(example_output.shape))

## Training

I use Huber loss because demand has occasional spikes and I do not want a few large errors to dominate training. AdamW, gradient clipping, learning-rate reduction and early stopping are used to keep the small model stable.

In [ ]:
@dataclass(frozen=True)
class TrainingResult:
    history: list[dict[str, float | int]]
    best_epoch: int
    best_validation_loss: float


def set_seed(
    seed: int,
    deterministic: bool = True,
) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )

        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    epochs: int,
    learning_rate: float,
    weight_decay: float,
    patience: int,
    gradient_clip: float,
    device: torch.device,
) -> TrainingResult:
    loss_function = nn.HuberLoss(delta=1.0)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=max(2, patience // 3),
    )

    best_state = copy.deepcopy(model.state_dict())
    best_loss = float("inf")
    best_epoch = 0
    remaining_patience = patience
    history: list[dict[str, float | int]] = []

    for epoch in range(1, epochs + 1):
        model.train()

        training_loss = 0.0
        training_items = 0

        for features, targets in train_loader:
            features = features.to(
                device,
                non_blocking=True,
            )

            targets = targets.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            predictions = model(features)
            loss = loss_function(predictions, targets)

            loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                gradient_clip,
            )

            optimizer.step()

            training_loss += loss.item() * len(features)
            training_items += len(features)

        model.eval()

        validation_loss = 0.0
        validation_items = 0

        with torch.inference_mode():
            for features, targets in validation_loader:
                features = features.to(
                    device,
                    non_blocking=True,
                )

                targets = targets.to(
                    device,
                    non_blocking=True,
                )

                loss = loss_function(
                    model(features),
                    targets,
                )

                validation_loss += loss.item() * len(features)
                validation_items += len(features)

        train_average = training_loss / training_items
        validation_average = (
            validation_loss / validation_items
        )

        scheduler.step(validation_average)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_average,
                "validation_loss": validation_average,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )

        print(
            f"Epoch {epoch:03d} | "
            f"train={train_average:.6f} | "
            f"validation={validation_average:.6f} | "
            f"lr={optimizer.param_groups[0]['lr']:.6g}"
        )

        if validation_average < best_loss - 1e-6:
            best_loss = validation_average
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            remaining_patience = patience
        else:
            remaining_patience -= 1

            if remaining_patience == 0:
                print(
                    "Early stopping at epoch",
                    epoch,
                )
                break

    model.load_state_dict(best_state)

    return TrainingResult(
        history=history,
        best_epoch=best_epoch,
        best_validation_loss=float(best_loss),
    )


def predict_model(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    predictions: list[np.ndarray] = []
    actuals: list[np.ndarray] = []

    model.eval()

    with torch.inference_mode():
        for features, targets in loader:
            output = model(
                features.to(
                    device,
                    non_blocking=True,
                )
            ).cpu().numpy()

            predictions.append(output)
            actuals.append(targets.numpy())

    return (
        np.concatenate(predictions),
        np.concatenate(actuals),
    )

In [ ]:
set_seed(SEED)

model = DemandTransformer(
    num_features=len(FEATURE_COLUMNS),
    target_dim=len(TARGET_COLUMNS),
    horizon=HORIZON,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
).to(DEVICE)

started = time.perf_counter()

training_result = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    gradient_clip=GRADIENT_CLIP,
    device=DEVICE,
)

runtime_seconds = time.perf_counter() - started

scaled_prediction, scaled_actual = predict_model(
    model,
    test_loader,
    DEVICE,
)

transformer_prediction = inverse_targets(
    scaled_prediction,
    prepared.target_scaler,
)

transformer_actual = inverse_targets(
    scaled_actual,
    prepared.target_scaler,
)

print("Best epoch:", training_result.best_epoch)
print(
    "Best validation loss:",
    training_result.best_validation_loss,
)
print(
    "Training runtime (minutes):",
    round(runtime_seconds / 60, 2),
)

## Baselines

A Transformer is only useful here if it adds something beyond simple forecasting rules. I therefore compare it with last-value persistence, the same hour one day ago, the same hour one week ago, a daily/weekly blend, Ridge regression and Extra Trees.

In [ ]:
def window_matrix(
    features: np.ndarray,
    target_starts: np.ndarray,
    lookback: int,
) -> np.ndarray:
    return np.stack(
        [
            features[
                start - lookback : start
            ].reshape(-1)
            for start in target_starts
        ]
    )


def target_matrix(
    targets: np.ndarray,
    target_starts: np.ndarray,
    horizon: int,
) -> np.ndarray:
    return np.stack(
        [
            targets[
                start : start + horizon
            ].reshape(-1)
            for start in target_starts
        ]
    )


class RidgeForecaster:
    def __init__(self, alpha: float = 10.0) -> None:
        self.alpha = float(alpha)
        self.model = Ridge(alpha=self.alpha)
        self.horizon = 0
        self.target_dim = 0
        self.lookback = 0

    def fit(
        self,
        features: np.ndarray,
        targets: np.ndarray,
        target_starts: np.ndarray,
        lookback: int,
        horizon: int,
    ):
        x = window_matrix(
            features,
            target_starts,
            lookback,
        )

        y = target_matrix(
            targets,
            target_starts,
            horizon,
        )

        self.horizon = horizon
        self.target_dim = targets.shape[1]
        self.lookback = lookback
        self.model.fit(x, y)

        return self

    def predict(
        self,
        features: np.ndarray,
        target_starts: np.ndarray,
        lookback: int,
    ) -> np.ndarray:
        x = window_matrix(
            features,
            target_starts,
            lookback,
        )

        prediction = self.model.predict(x)

        return prediction.reshape(
            -1,
            self.horizon,
            self.target_dim,
        )


class ExtraTreesForecaster:
    def __init__(
        self,
        n_estimators: int,
        min_samples_leaf: int,
        max_features: float,
        random_state: int,
    ) -> None:
        self.model = ExtraTreesRegressor(
            n_estimators=n_estimators,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=random_state,
            n_jobs=-1,
        )

        self.horizon = 0
        self.target_dim = 0
        self.lookback = 0

    def fit(
        self,
        features: np.ndarray,
        targets: np.ndarray,
        target_starts: np.ndarray,
        lookback: int,
        horizon: int,
    ):
        x = window_matrix(
            features,
            target_starts,
            lookback,
        )

        y = target_matrix(
            targets,
            target_starts,
            horizon,
        )

        self.horizon = horizon
        self.target_dim = targets.shape[1]
        self.lookback = lookback
        self.model.fit(x, y)

        return self

    def predict(
        self,
        features: np.ndarray,
        target_starts: np.ndarray,
        lookback: int,
    ) -> np.ndarray:
        prediction = self.model.predict(
            window_matrix(
                features,
                target_starts,
                lookback,
            )
        )

        return prediction.reshape(
            -1,
            self.horizon,
            self.target_dim,
        )


def select_ridge_alpha(
    features: np.ndarray,
    targets: np.ndarray,
    train_starts: np.ndarray,
    validation_starts: np.ndarray,
    lookback: int,
    horizon: int,
    candidates: tuple[float, ...],
):
    best_model = None
    best_alpha = float(candidates[0])
    best_loss = float("inf")

    validation_actual = target_matrix(
        targets,
        validation_starts,
        horizon,
    ).reshape(
        -1,
        horizon,
        targets.shape[1],
    )

    for alpha in candidates:
        candidate = RidgeForecaster(
            alpha=alpha
        ).fit(
            features,
            targets,
            train_starts,
            lookback,
            horizon,
        )

        prediction = candidate.predict(
            features,
            validation_starts,
            lookback,
        )

        loss = float(
            np.mean(
                np.abs(
                    prediction
                    - validation_actual
                )
            )
        )

        if loss < best_loss:
            best_loss = loss
            best_alpha = float(alpha)
            best_model = candidate

    return best_model, best_alpha, best_loss


def persistence_forecast(
    raw_targets: np.ndarray,
    target_starts: np.ndarray,
    horizon: int,
) -> np.ndarray:
    return np.stack(
        [
            np.repeat(
                raw_targets[start - 1][None, :],
                horizon,
                axis=0,
            )
            for start in target_starts
        ]
    )


def seasonal_forecast(
    raw_targets: np.ndarray,
    target_starts: np.ndarray,
    horizon: int,
    seasonality: int,
) -> np.ndarray:
    return np.stack(
        [
            raw_targets[
                start - seasonality :
                start - seasonality + horizon
            ]
            for start in target_starts
        ]
    )


def blended_seasonal_forecast(
    raw_targets: np.ndarray,
    target_starts: np.ndarray,
    horizon: int,
    seasonalities: tuple[int, ...],
) -> np.ndarray:
    forecasts = [
        seasonal_forecast(
            raw_targets,
            target_starts,
            horizon,
            seasonality,
        )
        for seasonality in seasonalities
    ]

    return np.mean(
        np.stack(forecasts, axis=0),
        axis=0,
    )

In [ ]:
ridge, ridge_alpha, ridge_validation_loss = select_ridge_alpha(
    prepared.features,
    prepared.targets,
    prepared.train_starts,
    prepared.validation_starts,
    LOOKBACK,
    HORIZON,
    RIDGE_ALPHAS,
)

ridge_prediction = inverse_targets(
    ridge.predict(
        prepared.features,
        prepared.test_starts,
        LOOKBACK,
    ),
    prepared.target_scaler,
)

extra_trees = ExtraTreesForecaster(
    n_estimators=EXTRA_TREES_ESTIMATORS,
    min_samples_leaf=EXTRA_TREES_MIN_SAMPLES_LEAF,
    max_features=EXTRA_TREES_MAX_FEATURES,
    random_state=SEED,
).fit(
    prepared.features,
    prepared.targets,
    prepared.train_starts,
    LOOKBACK,
    HORIZON,
)

extra_trees_prediction = inverse_targets(
    extra_trees.predict(
        prepared.features,
        prepared.test_starts,
        LOOKBACK,
    ),
    prepared.target_scaler,
)

evaluations = {
    "Transformer": transformer_prediction,
    "Ridge": ridge_prediction,
    "Extra Trees": extra_trees_prediction,
    "Persistence": persistence_forecast(
        prepared.raw_targets,
        prepared.test_starts,
        HORIZON,
    ),
    "Seasonal naive 24h": seasonal_forecast(
        prepared.raw_targets,
        prepared.test_starts,
        HORIZON,
        24,
    ),
    "Seasonal naive 168h": seasonal_forecast(
        prepared.raw_targets,
        prepared.test_starts,
        HORIZON,
        168,
    ),
    "Seasonal blend 24h+168h": blended_seasonal_forecast(
        prepared.raw_targets,
        prepared.test_starts,
        HORIZON,
        (24, 168),
    ),
}

print("Selected Ridge alpha:", ridge_alpha)
print("Ridge validation loss:", ridge_validation_loss)

## Evaluation

Metrics are calculated after converting the predictions back to event counts. I report MAE, RMSE, WAPE, sMAPE, bias, MASE and RMSSE, and I also look at how the Transformer's error changes across the 24-hour horizon.

In [ ]:
def scaled_error_denominators(
    training_actual: np.ndarray,
    seasonality: int,
) -> tuple[np.ndarray, np.ndarray]:
    differences = (
        training_actual[seasonality:]
        - training_actual[:-seasonality]
    )

    mae_scale = np.maximum(
        np.mean(np.abs(differences), axis=0),
        1e-8,
    )

    mse_scale = np.maximum(
        np.mean(differences**2, axis=0),
        1e-8,
    )

    return mae_scale, mse_scale


def regression_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
    target_names: tuple[str, ...],
    training_actual: np.ndarray,
    seasonality: int = 24,
) -> pd.DataFrame:
    mae_scale, mse_scale = scaled_error_denominators(
        training_actual,
        seasonality,
    )

    rows = []

    for index, target in enumerate(target_names):
        observed = actual[..., index].reshape(-1)
        forecast = predicted[..., index].reshape(-1)

        error = forecast - observed
        absolute_error = np.abs(error)
        denominator = np.maximum(
            np.abs(observed),
            1.0,
        )

        rows.append(
            {
                "target": target,
                "mae": float(
                    absolute_error.mean()
                ),
                "rmse": float(
                    np.sqrt(
                        np.mean(error**2)
                    )
                ),
                "wape": float(
                    absolute_error.sum()
                    / np.maximum(
                        np.abs(observed).sum(),
                        1.0,
                    )
                ),
                "smape": float(
                    np.mean(
                        2
                        * absolute_error
                        / np.maximum(
                            np.abs(observed)
                            + np.abs(forecast),
                            1.0,
                        )
                    )
                ),
                "mape": float(
                    np.mean(
                        absolute_error
                        / denominator
                    )
                ),
                "bias": float(
                    np.mean(error)
                ),
                "mase": float(
                    absolute_error.mean()
                    / mae_scale[index]
                ),
                "rmsse": float(
                    np.sqrt(
                        np.mean(error**2)
                        / mse_scale[index]
                    )
                ),
            }
        )

    return pd.DataFrame(rows)


def horizon_metrics(
    actual: np.ndarray,
    predicted: np.ndarray,
    target_names: tuple[str, ...],
) -> pd.DataFrame:
    rows = []

    for horizon_index in range(actual.shape[1]):
        for target_index, target in enumerate(target_names):
            observed = actual[
                :,
                horizon_index,
                target_index,
            ]

            forecast = predicted[
                :,
                horizon_index,
                target_index,
            ]

            error = forecast - observed
            absolute_error = np.abs(error)

            rows.append(
                {
                    "forecast_hour": horizon_index + 1,
                    "target": target,
                    "mae": float(
                        absolute_error.mean()
                    ),
                    "rmse": float(
                        np.sqrt(
                            np.mean(error**2)
                        )
                    ),
                    "bias": float(
                        error.mean()
                    ),
                }
            )

    return pd.DataFrame(rows)


training_actual = prepared.raw_targets[
    : prepared.fold.train_end
]

metric_frames = []

for name, prediction in evaluations.items():
    metrics = regression_metrics(
        transformer_actual,
        prediction,
        TARGET_COLUMNS,
        training_actual=training_actual,
        seasonality=24,
    )

    metrics.insert(0, "model", name)
    metric_frames.append(metrics)

comparison = pd.concat(
    metric_frames,
    ignore_index=True,
)

comparison["average_mae_for_model"] = (
    comparison.groupby("model")["mae"]
    .transform("mean")
)

comparison = comparison.sort_values(
    ["average_mae_for_model", "target"]
).reset_index(drop=True)

transformer_horizon = horizon_metrics(
    transformer_actual,
    transformer_prediction,
    TARGET_COLUMNS,
)

display(comparison)
display(transformer_horizon.head(10))

## Save the run

I save the model, scalers, predictions, metrics and plots to Drive. The raw and processed datasets stay out of the public repository.

In [ ]:
ARTIFACT_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

comparison.to_csv(
    ARTIFACT_DRIVE_DIR / "model_comparison.csv",
    index=False,
)

transformer_horizon.to_csv(
    ARTIFACT_DRIVE_DIR / "horizon_metrics.csv",
    index=False,
)

history_frame = pd.DataFrame(
    training_result.history
)

history_frame.to_csv(
    ARTIFACT_DRIVE_DIR / "training_history.csv",
    index=False,
)

(ARTIFACT_DRIVE_DIR / "training_history.json").write_text(
    json.dumps(
        training_result.history,
        indent=2,
    ),
    encoding="utf-8",
)

torch.save(
    model.state_dict(),
    ARTIFACT_DRIVE_DIR / "transformer.pt",
)

joblib.dump(
    prepared.feature_scaler,
    ARTIFACT_DRIVE_DIR / "feature_scaler.joblib",
)

joblib.dump(
    prepared.target_scaler,
    ARTIFACT_DRIVE_DIR / "target_scaler.joblib",
)

joblib.dump(
    ridge,
    ARTIFACT_DRIVE_DIR / "ridge.joblib",
)

joblib.dump(
    extra_trees,
    ARTIFACT_DRIVE_DIR / "extra_trees.joblib",
)

metadata = {
    "feature_columns": list(FEATURE_COLUMNS),
    "target_columns": list(TARGET_COLUMNS),
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "train_windows": int(
        len(prepared.train_starts)
    ),
    "validation_windows": int(
        len(prepared.validation_starts)
    ),
    "test_windows": int(
        len(prepared.test_starts)
    ),
    "device": str(DEVICE),
    "parameter_count": count_parameters(model),
    "best_epoch": training_result.best_epoch,
    "best_validation_loss": (
        training_result.best_validation_loss
    ),
    "runtime_seconds": runtime_seconds,
    "ridge_alpha": ridge_alpha,
    "model": {
        "d_model": D_MODEL,
        "nhead": NHEAD,
        "num_layers": NUM_LAYERS,
        "dim_feedforward": DIM_FEEDFORWARD,
        "dropout": DROPOUT,
    },
    "data_audit": audit,
}

(ARTIFACT_DRIVE_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

prediction_rows = []

for model_name, prediction in evaluations.items():
    for window_index, target_start in enumerate(
        prepared.test_starts
    ):
        for horizon_index in range(HORIZON):
            timestamp_index = (
                int(target_start)
                + horizon_index
            )

            for target_index, target in enumerate(
                TARGET_COLUMNS
            ):
                prediction_rows.append(
                    {
                        "model": model_name,
                        "window": window_index,
                        "forecast_hour": (
                            horizon_index + 1
                        ),
                        "timestamp": featured.iloc[
                            timestamp_index
                        ]["timestamp"],
                        "target": target,
                        "actual": transformer_actual[
                            window_index,
                            horizon_index,
                            target_index,
                        ],
                        "prediction": prediction[
                            window_index,
                            horizon_index,
                            target_index,
                        ],
                    }
                )

predictions_frame = pd.DataFrame(
    prediction_rows
)

predictions_frame.to_csv(
    ARTIFACT_DRIVE_DIR / "predictions.csv",
    index=False,
)

print("Artifacts saved to:", ARTIFACT_DRIVE_DIR)

In [ ]:
# Training history

figure = plt.figure(figsize=(9, 5))
axis = figure.add_subplot(111)

axis.plot(
    history_frame["epoch"],
    history_frame["train_loss"],
    label="Training",
)

axis.plot(
    history_frame["epoch"],
    history_frame["validation_loss"],
    label="Validation",
)

axis.set_xlabel("Epoch")
axis.set_ylabel("Huber loss")
axis.set_title("Transformer training history")
axis.legend()
axis.grid(alpha=0.25)

figure.tight_layout()
figure.savefig(
    ARTIFACT_DRIVE_DIR / "training_history.png",
    dpi=180,
)
plt.show()

In [ ]:
# Model comparison

model_summary = (
    comparison.groupby(
        "model",
        as_index=False,
    )["mae"]
    .mean()
    .sort_values("mae")
)

figure = plt.figure(figsize=(11, 5.5))
axis = figure.add_subplot(111)

axis.bar(
    model_summary["model"],
    model_summary["mae"],
)

axis.set_xlabel("Model")
axis.set_ylabel("Average MAE")
axis.set_title(
    "Average forecasting MAE across carts and orders"
)
axis.tick_params(axis="x", rotation=25)

figure.tight_layout()
figure.savefig(
    ARTIFACT_DRIVE_DIR / "model_comparison.png",
    dpi=180,
)
plt.show()

display(model_summary)

In [ ]:
# Error by forecast horizon

figure = plt.figure(figsize=(10, 5))
axis = figure.add_subplot(111)

for target, group in transformer_horizon.groupby(
    "target"
):
    axis.plot(
        group["forecast_hour"],
        group["mae"],
        marker="o",
        label=target,
    )

axis.set_xlabel("Forecast hour")
axis.set_ylabel("MAE")
axis.set_title(
    "Transformer error by forecast horizon"
)
axis.legend()
axis.grid(alpha=0.25)

figure.tight_layout()
figure.savefig(
    ARTIFACT_DRIVE_DIR / "horizon_mae.png",
    dpi=180,
)
plt.show()

In [ ]:
# First test-origin forecast examples

hours = np.arange(1, HORIZON + 1)

for target_index, target in enumerate(
    TARGET_COLUMNS
):
    figure = plt.figure(figsize=(10, 5))
    axis = figure.add_subplot(111)

    axis.plot(
        hours,
        transformer_actual[
            0,
            :,
            target_index,
        ],
        marker="o",
        label="Actual",
    )

    axis.plot(
        hours,
        transformer_prediction[
            0,
            :,
            target_index,
        ],
        marker="x",
        label="Transformer forecast",
    )

    axis.set_xlabel("Forecast hour")
    axis.set_ylabel(target.title())
    axis.set_title(
        f"24-hour {target} forecast"
    )
    axis.legend()
    axis.grid(alpha=0.25)

    figure.tight_layout()
    figure.savefig(
        ARTIFACT_DRIVE_DIR
        / f"forecast_{target}.png",
        dpi=180,
    )
    plt.show()

## Result summary

This final cell gives me the main numbers I need when writing the project README or dissertation results section.


In [ ]:
summary = (
    comparison.groupby("model", as_index=False)
    .agg(mean_mae=("mae", "mean"), mean_wape=("wape", "mean"))
    .sort_values("mean_mae")
    .reset_index(drop=True)
)

print("Model ranking on the held-out test period:")
display(summary)

transformer_results = comparison[comparison["model"] == "Transformer"][
    ["target", "mae", "rmse", "wape", "mase", "bias"]
].copy()

print("\nTransformer test metrics:")
display(transformer_results)

best_model = summary.iloc[0]["model"]
print(f"\nLowest average MAE: {best_model}")
print(f"Transformer best epoch: {training_result.best_epoch}")
print(f"Transformer parameters: {count_parameters(model):,}")
print(f"Training time: {runtime_seconds / 60:.1f} minutes")
